# Function Calling and Tool Usage

LLMs normally generate text. That is already powerful: they can summarize, explain, classify, reason over context, and write code-like output.

But many real applications need more than text. They need to look up data, calculate something reliably, create a record, check a system, or ask another service for information.

**Tools** give an LLM a structured way to request those actions.

## Learning objectives

By the end of this notebook, you will be able to:

- Understand what function calling is.
- Define tools that an LLM can request.
- Execute tool calls from Python.
- Send tool results back to the model.
- Build a small interactive assistant.

This notebook uses the official OpenAI Python SDK and simple mock tools. We will not use LangChain.

## 1. Why do we need tools?

LLMs are good at language and reasoning over the information that appears in the prompt. They can decide what information would help answer a question.

But an LLM cannot directly:

- access private databases
- call APIs
- perform reliable calculations
- create support tickets
- check live system state

Without tools:

```text
User question
   ↓
LLM
   ↓
Text answer only
```

With tools:

```text
User question
   ↓
LLM decides a tool is needed
   ↓
Python code executes the tool
   ↓
Tool result is sent back to the LLM
   ↓
Final answer
```

**Important:** The LLM does not execute the function. It only requests a function call. Our application code executes the function.

## 2. What is function calling?

Function calling is a mechanism where the model can return **structured tool calls** instead of only natural language.

The basic flow is:

1. The developer defines available tools.
2. Each tool has a name, description, and JSON schema for its parameters.
3. The model decides whether to call a tool.
4. The model produces arguments for the tool.
5. The application executes the tool.
6. The result is sent back to the model.
7. The model uses the result to write the final answer.

Think of it as a contract:

```text
Model: "I want to call get_order_status with order_id='A1001'."
Python app: runs get_order_status("A1001")
Python app: sends the result back to the model
Model: writes a helpful answer for the user
```

## 3. Setup
Run the following cell

In [1]:
import importlib
import json
import os
import sys

from dotenv import load_dotenv

load_dotenv(override=False)

# ---------------------------------------------------------------------------
# Route all LLM calls through the LiteLLM proxy — no OpenAI key required.
# ---------------------------------------------------------------------------
import utils.open_ai as _openai_module
importlib.reload(_openai_module)
from utils.open_ai import OpenAI

client = OpenAI()

MODEL = os.getenv("MODEL")

print("OpenAI client is ready (routing via LiteLLM proxy).")
print("Model:", MODEL)


OpenAI client is ready (routing via LiteLLM proxy).
Model: gpt-4.1-mini


## 4. First tool: `multiply`

We will start with a tiny tool so the flow is easy to see.

The Python function is normal Python. The model cannot run it directly. We will describe the function to the model using a tool schema, and then our Python code will execute it if the model requests it.

In [2]:
def multiply(a: float, b: float) -> float:
    return a * b


multiply_tool = {
    "type": "function",
    "function": {
        "name": "multiply",
        "description": "Multiply two numbers and return the product.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number.",
                },
                "b": {
                    "type": "number",
                    "description": "The second number.",
                },
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    },
}

### A helper for the first tool call

This helper:

- sends a user question to the model with the `multiply` tool available
- prints whether the model requested a tool call
- extracts the tool name and arguments
- executes the Python function
- sends the result back to the model
- prints the final response

Watch for this important idea: the model does not calculate directly. It asks our application to call `multiply`.

In [3]:
def run_multiply_example(question: str):
    messages = [
        {
            "role": "system",
            "content": "Use the multiply tool whenever multiplication is needed.",
        },
        {"role": "user", "content": question},
    ]

    first_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=[multiply_tool],
        tool_choice="auto",
    )

    assistant_message = first_response.choices[0].message
    tool_calls = assistant_message.tool_calls or []

    if not tool_calls:
        print("The model did not request a tool call.")
        print("Model answer:", assistant_message.content)
        return assistant_message.content

    print(f"The model requested {len(tool_calls)} tool call(s).")

    # Add the assistant message containing the tool call to the conversation.
    messages.append(assistant_message)

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print("Tool requested:", tool_name)
        print("Arguments:", arguments)

        if tool_name == "multiply":
            result = multiply(**arguments)
        else:
            result = {"error": f"Unknown tool: {tool_name}"}

        print("Python executed the tool and got:", result)

        # Send the tool result back to the model.
        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            }
        )

    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    final_answer = final_response.choices[0].message.content
    print("\nFinal answer:")
    print(final_answer)
    return final_answer

In [4]:
run_multiply_example("What is 17.5 multiplied by 23?")

The model requested 1 tool call(s).
Tool requested: multiply
Arguments: {'a': 17.5, 'b': 23}
Python executed the tool and got: 402.5

Final answer:
17.5 multiplied by 23 is 402.5.


'17.5 multiplied by 23 is 402.5.'

## 5. Tool schema explained

A tool schema tells the model what the tool is for and what arguments it can send.

```python
{
    "type": "function",              # This tool is a callable function
    "function": {
        "name": "multiply",          # The name the model will use
        "description": "...",        # Helps the model know when to use it
        "parameters": {              # JSON Schema for function arguments
            "type": "object",        # Arguments are passed as a JSON object
            "properties": {          # Allowed fields
                "a": {"type": "number"},
                "b": {"type": "number"}
            },
            "required": ["a", "b"]   # Fields the model must provide
        }
    }
}
```

Key parts:

- `type`: tells the API this is a function tool.
- `name`: the function name the model will request.
- `description`: helps the model choose the right tool.
- `parameters`: describes the expected arguments using JSON Schema.
- `properties`: lists the available argument names and types.
- `required`: lists arguments that must be present.

Good schemas make tool usage much more reliable.

## 6. Multiple tools

Now we will give the model several tools and let it choose among them.

Our mock business scenario is customer support. The assistant can:

- check an order status
- calculate a refund amount
- create a support ticket

These are mock tools. They use in-memory Python dictionaries, not real external APIs.

In [5]:
CUSTOMER_ORDERS = {
    "A1001": {"status": "shipped", "eta": "tomorrow", "customer": "Ana", "total": 120.00},
    "B2002": {"status": "processing", "eta": "Friday", "customer": "Bruno", "total": 75.50},
    "C3003": {"status": "delayed", "eta": "next Monday", "customer": "Carla", "total": 210.00},
}

SUPPORT_TICKETS = []


def get_order_status(order_id: str) -> dict:
    order = CUSTOMER_ORDERS.get(order_id)
    if not order:
        return {
            "found": False,
            "order_id": order_id,
            "error": "Order not found.",
        }

    return {
        "found": True,
        "order_id": order_id,
        "status": order["status"],
        "eta": order["eta"],
        "customer": order["customer"],
    }


def calculate_refund_amount(order_id: str) -> dict:
    order = CUSTOMER_ORDERS.get(order_id)
    if not order:
        return {
            "found": False,
            "order_id": order_id,
            "error": "Order not found. Refund cannot be calculated.",
        }

    if order["status"] == "delayed":
        refund = round(order["total"] * 0.20, 2)
        reason = "20% refund because the order is delayed."
    elif order["status"] == "shipped":
        refund = 0.0
        reason = "No automatic refund because the order has shipped."
    else:
        refund = 0.0
        reason = "No automatic refund while the order is still processing."

    return {
        "found": True,
        "order_id": order_id,
        "refund_amount": refund,
        "currency": "USD",
        "reason": reason,
    }


def create_support_ticket(customer_name: str, issue: str, priority: str) -> dict:
    ticket_id = f"TICKET-{len(SUPPORT_TICKETS) + 1:04d}"
    ticket = {
        "ticket_id": ticket_id,
        "customer_name": customer_name,
        "issue": issue,
        "priority": priority,
        "status": "open",
    }
    SUPPORT_TICKETS.append(ticket)
    return ticket

In [6]:
business_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Look up the status, ETA, and customer name for a customer order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID, such as A1001.",
                    }
                },
                "required": ["order_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_refund_amount",
            "description": "Calculate the mock refund amount for an order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order ID, such as C3003.",
                    }
                },
                "required": ["order_id"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_support_ticket",
            "description": "Create a customer support ticket for an issue that needs human follow-up.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "The customer's name.",
                    },
                    "issue": {
                        "type": "string",
                        "description": "A short description of the support issue.",
                    },
                    "priority": {
                        "type": "string",
                        "description": "The ticket priority. Use low, medium, or high.",
                    },
                },
                "required": ["customer_name", "issue", "priority"],
                "additionalProperties": False,
            },
        },
    },
]

TOOL_REGISTRY = {
    "get_order_status": get_order_status,
    "calculate_refund_amount": calculate_refund_amount,
    "create_support_ticket": create_support_ticket,
}

## 7. Generic tool-calling loop

The first example was hard-coded for one tool. Real applications need a reusable loop.

This function:

- sends a user message and available tools to the model
- detects tool calls
- executes all requested tool calls
- appends tool results to the conversation
- calls the model again
- returns the final answer

The code also uses small helper functions so it is easier to inspect what the SDK returns.

In [7]:
def tool_call_to_dict(tool_call):
    # Convert a tool call object into a plain dictionary for easier printing.
    if hasattr(tool_call, "model_dump"):
        return tool_call.model_dump()
    return dict(tool_call)


def run_tool_calling_conversation(user_message: str, tools: list, system_prompt: str = None):
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": user_message})

    first_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    assistant_message = first_response.choices[0].message
    tool_calls = assistant_message.tool_calls or []

    if not tool_calls:
        return assistant_message.content

    # The assistant message must be added before the tool results.
    # It contains the tool_call IDs that the API expects us to answer.
    messages.append(assistant_message)

    for tool_call in tool_calls:
        tool_name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            arguments = {}
            tool_result = {
                "error": "The model returned invalid JSON arguments.",
                "raw_arguments": tool_call.function.arguments,
            }
        else:
            tool_function = TOOL_REGISTRY.get(tool_name)

            if tool_function is None:
                tool_result = {"error": f"Unknown tool requested: {tool_name}"}
            else:
                try:
                    tool_result = tool_function(**arguments)
                except TypeError as exc:
                    tool_result = {
                        "error": "Tool arguments did not match the Python function.",
                        "details": str(exc),
                        "arguments": arguments,
                    }
                except Exception as exc:
                    tool_result = {
                        "error": "Tool execution failed.",
                        "details": str(exc),
                    }

        print("Tool call:")
        print(json.dumps(tool_call_to_dict(tool_call), indent=2))
        print("Tool result:")
        print(json.dumps(tool_result, indent=2))

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result),
            }
        )

    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )

    return final_response.choices[0].message.content

## 8. Demo: customer support assistant

Now we have a small interactive AI application pattern.

The assistant should use tools when it needs business data or needs to create a ticket.

Notice the system prompt:

```text
You are a helpful customer support assistant. Use tools when needed.
Do not invent order statuses, refund amounts or ticket IDs.
If you need information, call the appropriate tool.
```

This tells the assistant not to guess values that should come from tools.

In [8]:
customer_support_prompt = (
    "You are a helpful customer support assistant. "
    "Use tools when needed. "
    "Do not invent order statuses, refund amounts or ticket IDs. "
    "If you need information, call the appropriate tool."
)

In [9]:
examples = [
    "Where is order A1001?",
    "Can I get a refund for order C3003?",
    "Please create a high priority ticket for Ana. The issue is that her package arrived damaged.",
    "What is the status of order X9999?",
    "Where is order A1001? And please create a high priority ticket for Ana. The issue is that her package arrived damaged."
]

for example in examples:
    print("=" * 80)
    print("User:", example)
    answer = run_tool_calling_conversation(
        user_message=example,
        tools=business_tools,
        system_prompt=customer_support_prompt,
    )
    print("Assistant:", answer)

User: Where is order A1001?
Tool call:
{
  "id": "call_1cQEm9ahUe0BRymhKJE3Zydx",
  "type": "function",
  "function": {
    "name": "get_order_status",
    "arguments": "{\"order_id\":\"A1001\"}"
  }
}
Tool result:
{
  "found": true,
  "order_id": "A1001",
  "status": "shipped",
  "eta": "tomorrow",
  "customer": "Ana"
}
Assistant: Order A1001 has been shipped and is expected to arrive tomorrow. If you need any more information, feel free to ask!
User: Can I get a refund for order C3003?
Tool call:
{
  "id": "call_JapMq0u2GaK1AkfUQp1aAPKK",
  "type": "function",
  "function": {
    "name": "calculate_refund_amount",
    "arguments": "{\"order_id\":\"C3003\"}"
  }
}
Tool result:
{
  "found": true,
  "order_id": "C3003",
  "refund_amount": 42.0,
  "currency": "USD",
  "reason": "20% refund because the order is delayed."
}
Assistant: Yes, you can get a refund for order C3003. You are eligible for a 20% refund due to the delay in the order, which amounts to 42.0 USD. Would you like me to i

## 9. Tool usage vs normal prompting

Prompting and tool usage solve different problems.

Prompting controls how the model responds:

- tone
- format
- role
- constraints
- reasoning style

Tools expand what the model can actually do:

- look up private state
- call business logic
- calculate consistently
- create records
- retrieve controlled data

Prompting alone cannot reliably access external state. If an order status is not in the prompt, the model may guess. Function calling gives the model a structured interface to external capabilities.

## 10. Interactive loop

This is the simplest version of an interactive AI application in a notebook.

To avoid blocking `Run All`, the loop is turned off by default. Change `RUN_INTERACTIVE_CHAT` to `True` when you want to try it live.

In [10]:
def ask_customer_support_assistant(user_input: str) -> str:
    return run_tool_calling_conversation(
        user_message=user_input,
        tools=business_tools,
        system_prompt=customer_support_prompt,
    )


RUN_INTERACTIVE_CHAT = True

if RUN_INTERACTIVE_CHAT:
    print("Type 'exit' or 'quit' to stop.")

    while True:
        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Assistant: Goodbye!")
            break

        answer = ask_customer_support_assistant(user_input)
        print("Assistant:", answer)

Type 'exit' or 'quit' to stop.
Assistant: Goodbye!


## 11. Common problems and limitations

Function calling is powerful, but it is not magic.

Common issues:

- Bad tool descriptions lead to wrong tool choices.
- Missing required parameters may require follow-up questions.
- Tools can return errors.
- The model may call the wrong tool.
- The model may overuse tools.
- Tool outputs must be validated.
- Never expose unsafe tools without guardrails.
- Function calling does not remove hallucinations; it reduces them when answers depend on tool results.

Useful debugging questions:

```text
Did the model choose the right tool?
Did the model pass the right arguments?
Did our Python function validate those arguments?
Did we send the tool result back correctly?
Did the final answer stay grounded in the tool result?
```

## 12. Safety and design considerations

Some tools are safer than others.

Read-only tools are usually safer:

- `get_order_status`
- `search_faq`
- `calculate_price`

Write tools need more care:

- `send_email`
- `delete_file`
- `create_ticket`
- `refund_payment`

Good practices:

- Validate arguments before executing tools.
- Log tool calls.
- Handle errors gracefully.
- Use least privilege.
- Add confirmation steps before important write actions.
- Keep tool descriptions specific.
- Return structured error dictionaries from tools.

For example, a customer support assistant may be allowed to search orders automatically, but should ask for confirmation before creating a support ticket.

## 13. Mini exercises

Try these exercises during class or as homework.

### Exercise 1

Add a new tool:

```python
def estimate_shipping_cost(country: str, weight_kg: float) -> dict
```

The tool should return a mock shipping estimate.

Ask the model:

```text
How much would it cost to ship a 3 kg package to Germany?
```

In [11]:
# Your Exercise 1 code here.

### Exercise 2

Improve the `create_support_ticket` tool so that it only accepts priority values:

```text
low, medium, high
```

If the model sends another value, return an error dictionary.

In [12]:
# Your Exercise 2 code here.

### Exercise 3

Modify the assistant so that ticket creation requires confirmation.

Example:

```text
User: Create a ticket for Bruno because his order is late.
Assistant: I can create a ticket for Bruno with issue "...". Please confirm.
```

Only after the user confirms should the application call `create_support_ticket`.

In [13]:
# Your Exercise 3 code here.

### Exercise 4

Add a tool:

```python
def search_faq(query: str) -> dict
```

Use a small in-memory FAQ dictionary.

Let the assistant answer questions using this FAQ tool.

In [14]:
# Your Exercise 4 code here.

## 14. Final summary

Function calling lets LLMs request external actions.

Main ideas:

- Tools are Python functions described with schemas.
- The model chooses the tool and arguments.
- The application executes the function.
- Tool results are sent back to the model.
- The final answer should be grounded in the tool result.
- This enables interactive AI applications.

Frameworks like LangChain can automate parts of this loop, but the underlying mechanism is the same:

```text
Model requests tool
   ↓
Application executes tool
   ↓
Application returns result
   ↓
Model writes final answer
```

In production, the same pattern can connect an assistant to databases, APIs, ticketing systems, calculators, search tools, and internal business workflows.